# SFT vs DPO With Frozen Reference and Reload Proof

**Question:** After genuine SFT and DPO updates, does held-out preference likelihood change against the exact frozen SFT reference?

Kerem Ozcan | IBM AI Engineering portfolio | Prepared with AI assistance

Original experiment code developed from course concepts. This notebook contains no IBM assessment solutions or supplied course model predictions.

## Design and evaluation boundary

An extension of the instruction-tuning project, using Meta OPT-350M and cleaned Argilla UltraFeedback preference pairs. Keep aligned single-turn pairs and group normalized prompts before80/10/10 splitting. Select256 eligible training/64 validation/64 test pairs. SFT on chosen answers for one pass, then one DPO pass (128 updates per stage, batch size2). Mask prompts/padding, reject identical truncated pairs, disable OPT/LoRA dropout consistently and cache frozen SFT reference log probabilities. Include the SFT weights in checkpoint selection: retain them if candidate DPO validation loss does not improve. Independently reload both SFT and selected-DPO adapters onto fresh bases at the same revision. Report raw and length-normalized pair likelihood rates, chosen-response NLL and reference-adjusted DPO loss, plus fresh paired examples.

Small SFT/preference budgets, one seed, synthetic preference labels and truncated responses. Pairwise log-probability rates and implicit DPO margins are not generated-response win rates, task correctness or human-validated quality. This is a separate alignment experiment on the same base architecture, not reuse of the earlier CodeAlpaca adapter.

## Runtime

Kaggle internet is required for provider-hosted inputs/models. Local verification uses already installed isolated environments. `SECOND_BATCH_FULL=0` selects a small local correctness pilot for the neural projects; the published Kaggle version uses the documented full experiment budget. Data and weights stay in scratch storage; only JSON evidence, plots and notebook outputs are exported.

In [ ]:
import os
import sys
import subprocess
import gzip
import shutil
from pathlib import Path
os.environ["USE_TF"]="0"
os.environ["TOKENIZERS_PARALLELISM"]="false"
os.environ["OMP_NUM_THREADS"]="1"
ON_KAGGLE=Path("/kaggle/working").is_dir()
if ON_KAGGLE and True:
    subprocess.run([sys.executable,"-m","pip","install","-q",*['transformers==4.45.2', 'peft==0.11.1', 'accelerate==1.15.0', 'pyarrow>=17,<27']],check=True)
import matplotlib.pyplot as plt
import numpy as np
import json
FULL_RUN=os.environ.get("SECOND_BATCH_FULL","1")=="1"
LOCAL_ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/"provenance/source-manifest.json").exists()),Path.cwd())
WORK=Path("/kaggle/working") if ON_KAGGLE else LOCAL_ROOT/".lab-cache/second-batch-notebooks"/'preference_alignment'/"exports"
SCRATCH=Path("/kaggle/temp/ibm-second-batch") if ON_KAGGLE else LOCAL_ROOT/".lab-cache/second-batch-notebooks"/'preference_alignment'/"scratch"
WORK.mkdir(parents=True,exist_ok=True);SCRATCH.mkdir(parents=True,exist_ok=True)
DATA_ROOT=SCRATCH/"inputs" if ON_KAGGLE else LOCAL_ROOT/".lab-cache/second-batch-data"
RUN_OUTPUT=SCRATCH/'preference_alignment';OUTPUT=WORK/'preference_alignment';OUTPUT.mkdir(parents=True,exist_ok=True)


## Inputs and provenance

[Argilla UltraFeedback Binarized Preferences Cleaned](https://huggingface.co/datasets/argilla/ultrafeedback-binarized-preferences-cleaned), MIT per its dataset card, pinned to revision770076f077c4c5e298498fa32f804857f46d5134. Base: [Meta OPT-350M](https://huggingface.co/facebook/opt-350m), with its research/noncommercial terms. [DPO paper](https://arxiv.org/abs/2305.18290). No supplied course checkpoint, pretrained base weights or dataset archive is bundled.

Provider downloads must match the pinned SHA-256 values below. Model weights are fetched from the original provider; no submitted lab checkpoint is reused.

In [ ]:
"""Small, standalone data acquisition helpers embedded in the Kaggle releases."""

import hashlib
import shutil
import tarfile
import urllib.request
from pathlib import Path


def verified_download(url: str, target: Path, expected_sha256: str) -> Path:
    """Download to a temporary file and accept only the reviewed dataset bytes."""
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.is_file() and hashlib.sha256(target.read_bytes()).hexdigest() == expected_sha256:
        return target
    temporary = target.with_suffix(target.suffix + ".download")
    try:
        with urllib.request.urlopen(url, timeout=60) as response, temporary.open("wb") as stream:
            shutil.copyfileobj(response, stream)
        if hashlib.sha256(temporary.read_bytes()).hexdigest() != expected_sha256:
            raise ValueError("Downloaded data changed; review its provenance before running.")
        temporary.replace(target)
    finally:
        temporary.unlink(missing_ok=True)
    return target


def unpack_regular_tar(archive: Path, destination: Path) -> None:
    """Reject links, special files and traversal before writing any archive member."""
    root = destination.resolve()
    with tarfile.open(archive) as bundle:
        members = bundle.getmembers()
        for member in members:
            path = root / member.name
            if (
                Path(member.name).is_absolute()
                or not path.resolve().is_relative_to(root)
                or not (member.isfile() or member.isdir())
            ):
                raise ValueError("Unsafe dataset archive member.")
        for member in members:
            path = root / member.name
            if member.isdir():
                path.mkdir(parents=True, exist_ok=True)
            else:
                path.parent.mkdir(parents=True, exist_ok=True)
                with bundle.extractfile(member) as source, path.open("wb") as target:
                    shutil.copyfileobj(source, target)


In [ ]:
INPUTS=[{'file': 'preference/cleaned.parquet', 'url': 'https://huggingface.co/datasets/argilla/ultrafeedback-binarized-preferences-cleaned/resolve/770076f077c4c5e298498fa32f804857f46d5134/data/train-00000-of-00001.parquet', 'sha256': '27dc59041d73654aecfa7685b6f839a0bbb2014d640ee92eefa047b262d1f1ca', 'bytes': 143257393}]
for entry in INPUTS:
    path=verified_download(entry["url"],DATA_ROOT/entry["file"],entry["sha256"])
    if path.suffix==".gz":
        with gzip.open(path,"rb") as source,path.with_suffix("").open("wb") as target:
            shutil.copyfileobj(source,target)


## Standalone implementation

Expand this cell to inspect data partitions, actual sample accounting, training and selected-checkpoint reload checks.

In [ ]:
"""Shared provenance and evaluation helpers for the selected portfolio projects."""

from __future__ import annotations

import hashlib
import importlib.metadata
import json
import platform
import random
from collections import Counter, defaultdict
from datetime import UTC, datetime
from pathlib import Path


def file_sha256(path: Path) -> str:
    """Hash an input/checkpoint without reading the entire file into memory."""
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def write_json(path: Path, value: dict | list) -> None:
    """Publish a complete JSON artifact atomically."""
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + "\n", encoding="utf-8"
    )
    temporary.replace(path)


def runtime(packages: list[str]) -> dict:
    """Capture versions without collecting environment variables or host paths."""
    return {
        "utc": datetime.now(UTC).isoformat(),
        "python": platform.python_version(),
        "platform": platform.system(),
        "packages": {name: importlib.metadata.version(name) for name in packages},
    }


def source_identity(project_file: str | None, embedded_digest: str | None = None) -> dict:
    """Bind module runs and generated notebooks to their authored source files."""
    project_digest = file_sha256(Path(project_file)) if project_file else embedded_digest
    common_digest = (
        file_sha256(Path(__file__))
        if "__file__" in globals()
        else globals().get("COMMON_SOURCE_SHA256")
    )
    if not project_digest or not common_digest:
        raise ValueError("Project source identity is missing.")
    return {"project_sha256": project_digest, "common_sha256": common_digest}


def index_images(
    root: Path, class_names: list[str], directories: dict[str, str] | None = None
) -> list[dict]:
    """Index decoded RGB content as well as bytes, retaining only relative filenames."""
    from PIL import Image, ImageOps

    rows = []
    directories = directories or {"unsplit": "."}
    for role, directory in directories.items():
        for label, name in enumerate(class_names):
            folder = root / directory / name
            if not folder.is_dir():
                raise ValueError(f"Missing image class directory: {directory}/{name}")
            for path in sorted(folder.rglob("*")):
                if path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
                    continue
                with Image.open(path) as original:
                    image = ImageOps.exif_transpose(original).convert("RGB")
                    pixels = str(image.size).encode() + image.tobytes()
                rows.append(
                    {
                        "file": path.relative_to(root).as_posix(),
                        "label": label,
                        "role": role,
                        "file_sha256": file_sha256(path),
                        "pixel_sha256": hashlib.sha256(pixels).hexdigest(),
                    }
                )
    if not rows:
        raise ValueError("No images found.")
    return rows


def clean_image_splits(rows: list[dict]) -> tuple[list[dict], list[dict]]:
    """Remove exact duplicates and conflicting labels, preserving held-out copies first."""
    groups = defaultdict(list)
    for row in rows:
        groups[row["pixel_sha256"]].append(row)
    retained, removed = [], []
    priority = {"test": 0, "validation": 1, "train": 2}
    for group in groups.values():
        if len({row["label"] for row in group}) > 1:
            removed.extend(
                {"file": row["file"], "reason": "conflicting_pixel_labels"} for row in group
            )
            continue
        ordered = sorted(group, key=lambda row: (priority[row["role"]], row["file"]))
        retained.append(ordered[0])
        removed.extend(
            {"file": row["file"], "reason": "exact_pixel_duplicate"} for row in ordered[1:]
        )
    return sorted(retained, key=lambda row: row["file"]), removed


def balanced_subset(rows: list[dict], per_class: int | None, seed: int) -> list[dict]:
    """Select a reproducible pilot without changing the complete split manifest."""
    if per_class is None:
        return rows
    rng = random.Random(seed)
    result = []
    for label in sorted({row["label"] for row in rows}):
        group = sorted((row for row in rows if row["label"] == label), key=lambda row: row["file"])
        rng.shuffle(group)
        result.extend(group[:per_class])
    return sorted(result, key=lambda row: row["file"])


def split_counts(rows: list[dict]) -> dict:
    """Describe the actual records, including class support."""
    return {
        role: dict(Counter(str(row["label"]) for row in rows if row["role"] == role))
        for role in ["train", "validation", "test"]
    }


def classification_metrics(labels: list[int], probabilities: list[float]) -> dict:
    """Evaluate exactly the predictions supplied, never the size of an upstream dataset."""
    import numpy as np
    from sklearn.metrics import (
        accuracy_score,
        classification_report,
        confusion_matrix,
        f1_score,
        roc_auc_score,
    )

    if not labels or len(labels) != len(probabilities):
        raise ValueError("Predictions and labels must have the same nonzero length.")
    probabilities = np.asarray(probabilities)
    if not np.isfinite(probabilities).all() or ((probabilities < 0) | (probabilities > 1)).any():
        raise ValueError("Expected finite probabilities in [0, 1].")
    predicted = (probabilities >= 0.5).astype(int)
    return {
        "samples": len(labels),
        "accuracy": float(accuracy_score(labels, predicted)),
        "macro_f1": float(
            f1_score(labels, predicted, labels=[0, 1], average="macro", zero_division=0)
        ),
        "roc_auc": float(roc_auc_score(labels, probabilities)) if len(set(labels)) == 2 else None,
        "confusion_matrix": confusion_matrix(labels, predicted, labels=[0, 1]).tolist(),
        "per_class": classification_report(
            labels, predicted, labels=[0, 1], output_dict=True, zero_division=0
        ),
    }

COMMON_SOURCE_SHA256='ded7059d91e2123ef7bc9be2823a1d220d905041f45b9dbe83bac59d27547853'
PROJECT_SOURCE_SHA256='832469907bdec27e5aa569c23917626af0dbc2df30c116ef2b638ce92bb80132'
"""Extend instruction tuning with real SFT and DPO updates against a frozen SFT reference."""


import gc
import hashlib
import json
import random
import time
from pathlib import Path

import torch



def instruction_prompt(text: str) -> str:
    return f"### Instruction:\n{text.strip()}\n\n### Response:\n"


def load_base(model_id: str, revision: str | None, allow_downloads: bool):
    """Pin the base revision and disable OPT dropout consistently for policy/reference scores."""
    from transformers import AutoConfig, AutoModelForCausalLM

    config = AutoConfig.from_pretrained(
        model_id, revision=revision, local_files_only=not allow_downloads
    )
    config.dropout = 0.0
    config.attention_dropout = 0.0
    resolved = getattr(config, "_commit_hash", None)
    if not resolved:
        raise ValueError("An immutable base revision is required.")
    return AutoModelForCausalLM.from_pretrained(
        model_id, revision=resolved, config=config, local_files_only=not allow_downloads
    )


def preference_partitions(path: Path, seed: int) -> dict:
    """Group normalized prompts before splitting; keep aligned single-turn response pairs."""
    import pyarrow.parquet as pq

    table = pq.read_table(path, columns=["prompt", "chosen", "rejected"])
    groups = {}
    for row in table.to_pylist():
        chosen, rejected = row["chosen"], row["rejected"]
        if len(chosen) != 2 or len(rejected) != 2:
            continue
        if [r["role"] for r in chosen] != ["user", "assistant"] or [
            r["role"] for r in rejected
        ] != ["user", "assistant"]:
            continue
        prompt = row["prompt"].strip()
        if chosen[0]["content"].strip() != prompt or rejected[0]["content"].strip() != prompt:
            continue
        yes, no = chosen[-1]["content"].strip(), rejected[-1]["content"].strip()
        if not prompt or not yes or not no or yes == no:
            continue
        group = hashlib.sha256(" ".join(prompt.split()).casefold().encode()).hexdigest()
        identifier = hashlib.sha256(
            json.dumps([prompt, yes, no], ensure_ascii=False).encode()
        ).hexdigest()
        groups.setdefault(group, {})[identifier] = {
            "id": identifier,
            "group": group,
            "prompt": prompt,
            "chosen": yes,
            "rejected": no,
        }
    keys = sorted(groups)
    random.Random(seed).shuffle(keys)
    a, b = int(len(keys) * 0.8), int(len(keys) * 0.9)
    return {
        role: [r for key in selection for r in groups[key].values()]
        for role, selection in [("train", keys[:a]), ("validation", keys[a:b]), ("test", keys[b:])]
    }


def encode_preference(row: dict, tokenizer, maximum: int = 256) -> dict:
    """Preserve the shared prompt, supervise responses only and reject identical truncations."""
    prefix = tokenizer.encode(instruction_prompt(row["prompt"]), add_special_tokens=True)
    if len(prefix) > maximum // 2:
        raise ValueError("Prompt exceeds the reserved prompt budget.")
    result = {}
    for side in ["chosen", "rejected"]:
        answer = tokenizer.encode(row[side], add_special_tokens=False) + [tokenizer.eos_token_id]
        limit = maximum - len(prefix)
        truncated = len(answer) > limit
        answer = answer[:limit]
        result[side] = {
            "input_ids": prefix + answer,
            "labels": [-100] * len(prefix) + answer,
            "truncated": truncated,
        }
    if result["chosen"]["input_ids"] == result["rejected"]["input_ids"]:
        raise ValueError("Responses become identical after truncation.")
    return {**result, "id": row["id"], "group": row["group"]}


def response_logps(model, examples: list[dict], pad_id: int, device):
    """Sum only supervised next-token log probabilities; retain token counts for NLL."""
    width = max(len(e["input_ids"]) for e in examples)
    ids = torch.full((len(examples), width), pad_id, dtype=torch.long, device=device)
    labels = torch.full_like(ids, -100)
    attention = torch.zeros_like(ids)
    for i, row in enumerate(examples):
        n = len(row["input_ids"])
        ids[i, :n] = torch.tensor(row["input_ids"], device=device)
        labels[i, :n] = torch.tensor(row["labels"], device=device)
        attention[i, :n] = 1
    logits = model(input_ids=ids, attention_mask=attention).logits[:, :-1]
    following = labels[:, 1:]
    mask = following.ne(-100)
    values = logits.log_softmax(-1).gather(-1, following.clamp_min(0).unsqueeze(-1)).squeeze(-1)
    counts = mask.sum(1)
    if bool((counts == 0).any()):
        raise ValueError("Every response needs supervised next-token targets.")
    return (values * mask).sum(1), counts


def dpo_objective(chosen, rejected, reference_chosen, reference_rejected, beta: float = 0.1):
    """Canonical sigmoid DPO with fixed reference sequence log probabilities."""
    margin = beta * ((chosen - rejected) - (reference_chosen - reference_rejected))
    return -torch.nn.functional.logsigmoid(margin).mean()


def pair_scores(model, examples: list[dict], pad_id: int, device) -> list[dict]:
    model.eval()
    result = []
    with torch.no_grad():
        for start in range(0, len(examples), 2):
            batch = examples[start : start + 2]
            flat = [r[side] for r in batch for side in ["chosen", "rejected"]]
            values, counts = response_logps(model, flat, pad_id, device)
            values = values.cpu().tolist()
            counts = counts.cpu().tolist()
            for i, row in enumerate(batch):
                result.append(
                    {
                        "id": row["id"],
                        "chosen": values[2 * i],
                        "rejected": values[2 * i + 1],
                        "chosen_tokens": counts[2 * i],
                        "rejected_tokens": counts[2 * i + 1],
                    }
                )
    return result


def summarize(scores: list[dict], reference: list[dict] | None = None) -> dict:
    nll = -sum(r["chosen"] for r in scores) / sum(r["chosen_tokens"] for r in scores)
    result = {
        "pairs": len(scores),
        "chosen_response_nll": nll,
        "raw_sequence_preference_rate": sum(r["chosen"] > r["rejected"] for r in scores)
        / len(scores),
        "length_normalized_preference_rate": sum(
            r["chosen"] / r["chosen_tokens"] > r["rejected"] / r["rejected_tokens"] for r in scores
        )
        / len(scores),
    }
    if reference is not None:
        if [r["id"] for r in scores] != [r["id"] for r in reference]:
            raise ValueError("Reference pairs are misaligned.")
        margins = [
            0.1 * ((r["chosen"] - r["rejected"]) - (s["chosen"] - s["rejected"]))
            for r, s in zip(scores, reference, strict=True)
        ]
        result["reference_adjusted_dpo_loss"] = sum(
            float(torch.nn.functional.softplus(torch.tensor(-m))) for m in margins
        ) / len(margins)
        result["reference_adjusted_margin_positive_rate"] = sum(m > 0 for m in margins) / len(
            margins
        )
    return result


def adapter_digest(model) -> str:
    """Check that optimization changes actual trainable tensor bytes."""
    digest = hashlib.sha256()
    for name, value in model.named_parameters():
        if value.requires_grad:
            digest.update(name.encode())
            digest.update(value.detach().cpu().numpy().tobytes())
    return digest.hexdigest()


def generate_examples(model, tokenizer, records: list[dict], device) -> list[str]:
    model.eval()
    outputs = []
    with torch.no_grad():
        for row in records[:6]:
            inputs = tokenizer(instruction_prompt(row["prompt"]), return_tensors="pt").to(device)
            ids = model.generate(
                **inputs,
                max_new_tokens=64,
                do_sample=False,
                num_beams=1,
                pad_token_id=tokenizer.pad_token_id,
            )
            outputs.append(
                tokenizer.decode(ids[0, inputs["input_ids"].shape[1] :], skip_special_tokens=True)
            )
    return outputs


def run(
    data_file: Path,
    output: Path,
    *,
    full: bool = False,
    seed: int = 42,
    allow_downloads: bool = False,
) -> dict:
    """Use a fresh SFT baseline, frozen reference scores and independently reload selected DPO."""
    from peft import LoraConfig, PeftModel, TaskType, get_peft_model
    from transformers import AutoTokenizer

    torch.set_num_threads(1)
    torch.manual_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_id = "facebook/opt-350m"
    model = load_base(model_id, None, allow_downloads).to(device)
    revision = getattr(model.config, "_commit_hash", None)
    if not revision:
        raise ValueError("An immutable base revision is required.")
    tokenizer = AutoTokenizer.from_pretrained(
        model_id, revision=revision, local_files_only=not allow_downloads
    )
    parts = preference_partitions(data_file, seed)
    selected, encoded, rejected = {}, {}, {}
    for role, rows in parts.items():
        selected[role], encoded[role], rejected[role] = [], [], 0
        limit = (256 if role == "train" else 64) if full else (16 if role == "train" else 8)
        for row in rows:
            try:
                example = encode_preference(row, tokenizer)
            except ValueError:
                rejected[role] += 1
                continue
            selected[role].append(row)
            encoded[role].append(example)
            if len(selected[role]) == limit:
                break
        if len(selected[role]) != limit:
            raise ValueError(f"Insufficient eligible {role} pairs.")
    manifest = {
        "partitions": {
            role: [{"id": r["id"], "group": r["group"]} for r in rows]
            for role, rows in selected.items()
        },
        "available_grouped_counts": {r: len(v) for r, v in parts.items()},
        "rejected_before_selection": rejected,
    }
    write_json(output / "split-manifest.json", manifest)
    del parts
    started = time.perf_counter()
    results = {
        "base": summarize(pair_scores(model, encoded["test"], tokenizer.pad_token_id, device))
    }
    generations = {"base": generate_examples(model, tokenizer, selected["test"], device)}
    model = get_peft_model(
        model,
        LoraConfig(
            r=8,
            lora_alpha=16,
            target_modules=["q_proj", "v_proj"],
            lora_dropout=0.0,
            task_type=TaskType.CAUSAL_LM,
        ),
    )
    initial = adapter_digest(model)
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=2e-4)
    order = list(range(len(encoded["train"])))
    random.Random(seed).shuffle(order)
    sft_steps = 0
    for start in range(0, len(order), 2):
        model.train()
        batch = [encoded["train"][i]["chosen"] for i in order[start : start + 2]]
        values, tokens = response_logps(model, batch, tokenizer.pad_token_id, device)
        loss = -values.sum() / tokens.sum()
        if not bool(torch.isfinite(loss)):
            raise ValueError("Non-finite SFT loss.")
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        sft_steps += 1
    if adapter_digest(model) == initial:
        raise AssertionError("SFT did not update the adapter.")
    sft_dir = output / "sft-adapter"
    model.save_pretrained(sft_dir, safe_serialization=True)
    reference = {
        role: pair_scores(model, examples, tokenizer.pad_token_id, device)
        for role, examples in encoded.items()
    }
    write_json(output / "reference-logps.json", reference)
    results["sft"] = summarize(reference["test"], reference["test"])
    generations["sft"] = generate_examples(model, tokenizer, selected["test"], device)
    probe = tokenizer(
        instruction_prompt(selected["validation"][0]["prompt"]), return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        expected = model(**probe).logits[:, -1].cpu()
    del model, optimizer
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    model = (
        PeftModel.from_pretrained(
            load_base(model_id, revision, allow_downloads),
            sft_dir,
            is_trainable=True,
        )
        .to(device)
        .eval()
    )
    with torch.no_grad():
        actual = model(**probe).logits[:, -1].cpu()
    torch.testing.assert_close(expected, actual, rtol=1e-5, atol=1e-5)
    sft_reload = float((expected - actual).abs().max())
    before_dpo = adapter_digest(model)
    best_dir = output / "selected-dpo-adapter"
    model.save_pretrained(best_dir, safe_serialization=True)
    initial_val = summarize(reference["validation"], reference["validation"])[
        "reference_adjusted_dpo_loss"
    ]
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=5e-5)
    dpo_steps = 0
    for start in range(0, len(order), 2):
        model.train()
        indices = order[start : start + 2]
        flat = [encoded["train"][i][side] for i in indices for side in ["chosen", "rejected"]]
        values, _ = response_logps(model, flat, tokenizer.pad_token_id, device)
        refs = torch.tensor(
            [[reference["train"][i][side] for side in ["chosen", "rejected"]] for i in indices],
            device=device,
        )
        loss = dpo_objective(values[::2], values[1::2], refs[:, 0], refs[:, 1])
        if not bool(torch.isfinite(loss)):
            raise ValueError("Non-finite DPO loss.")
        optimizer.zero_grad()
        loss.backward()
        nn_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        if not bool(torch.isfinite(nn_norm)):
            raise ValueError("Non-finite DPO gradients.")
        optimizer.step()
        dpo_steps += 1
    changed = adapter_digest(model) != before_dpo
    if not changed:
        raise AssertionError("DPO did not update the adapter.")
    candidate_val = summarize(
        pair_scores(model, encoded["validation"], tokenizer.pad_token_id, device),
        reference["validation"],
    )["reference_adjusted_dpo_loss"]
    selected_stage = "dpo" if candidate_val < initial_val else "sft-retained"
    if selected_stage == "dpo":
        model.save_pretrained(best_dir, safe_serialization=True)
    model.load_adapter(best_dir, adapter_name="selected", is_trainable=False)
    model.set_adapter("selected")
    model.eval()
    with torch.no_grad():
        expected = model(**probe).logits[:, -1].cpu()
    del model, optimizer
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    model = (
        PeftModel.from_pretrained(
            load_base(model_id, revision, allow_downloads),
            best_dir,
        )
        .to(device)
        .eval()
    )
    with torch.no_grad():
        actual = model(**probe).logits[:, -1].cpu()
    torch.testing.assert_close(expected, actual, rtol=1e-5, atol=1e-5)
    results["selected_dpo"] = summarize(
        pair_scores(model, encoded["test"], tokenizer.pad_token_id, device), reference["test"]
    )
    generations["selected_dpo"] = generate_examples(model, tokenizer, selected["test"], device)
    report = {
        "project": "preference_alignment",
        "profile": "bounded-sft-dpo" if full else "bounded-pilot",
        "seed": seed,
        "base_model": model_id,
        "base_revision": revision,
        "device": str(device),
        "data_sha256": file_sha256(data_file),
        "split_manifest_sha256": file_sha256(output / "split-manifest.json"),
        "selected_split_counts": {r: len(v) for r, v in selected.items()},
        "sft_optimizer_steps": sft_steps,
        "dpo_optimizer_steps": dpo_steps,
        "sft_adapter_sha256": file_sha256(sft_dir / "adapter_model.safetensors"),
        "selected_adapter_sha256": file_sha256(best_dir / "adapter_model.safetensors"),
        "reference_logps_sha256": file_sha256(output / "reference-logps.json"),
        "selected_stage": selected_stage,
        "sft_reload_max_abs_difference": sft_reload,
        "dpo_reload_max_abs_difference": float((expected - actual).abs().max()),
        "reload_verified": True,
        "dpo_training_changed_adapter": changed,
        "validation_losses": {"sft_reference": initial_val, "dpo_candidate": candidate_val},
        "models": results,
        "truncated_response_counts": {
            r: sum(e[side]["truncated"] for e in rows for side in ["chosen", "rejected"])
            for r, rows in encoded.items()
        },
        "examples": [
            {"prompt": row["prompt"], **{stage: values[i] for stage, values in generations.items()}}
            for i, row in enumerate(selected["test"][:6])
        ],
        "limitations": "Small preference/SFT budgets, one seed and synthetic preference labels. Pairwise reference likelihood and DPO margins are not generated-response win rates or human-validated quality. Long responses are truncated; identical truncated pairs are excluded. The SFT checkpoint is retained when validation does not favor DPO.",
        "seconds": time.perf_counter() - started,
        "runtime": runtime(["torch", "transformers", "peft", "pyarrow"]),
        "source": source_identity(
            globals().get("__file__"), globals().get("PROJECT_SOURCE_SHA256")
        ),
    }
    write_json(output / "metrics.json", report)
    return report

In [ ]:
report=run(DATA_ROOT/'preference/cleaned.parquet', RUN_OUTPUT, full=FULL_RUN, allow_downloads=ON_KAGGLE)
for artifact in RUN_OUTPUT.glob("*.json"):
    shutil.copyfile(artifact,OUTPUT/artifact.name)
print("Executed profile:",report["profile"])
print("Actual runtime:",report["runtime"])
print("Source identity:",report["source"])


## Freshly measured results

These values are generated by this execution. Keep unsuccessful baselines and the stated evaluation limits visible.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4))
names=list(report["models"])
axes[0].bar(names,[report["models"][n]["chosen_response_nll"] for n in names],color="#138477")
axes[1].bar(names,[report["models"][n]["length_normalized_preference_rate"] for n in names],color="#60758a")
axes[0].set(ylabel="Chosen response token NLL",title="Reference likelihood; lower is better")
axes[1].set(ylabel="Held-out pair likelihood preference rate",ylim=(0,1),title="Not a generated-response win rate")
fig.tight_layout();fig.savefig(OUTPUT/"results.png",dpi=160);plt.show()
print("Actual pairs:",report["selected_split_counts"],"SFT/DPO updates:",report["sft_optimizer_steps"],report["dpo_optimizer_steps"])
print("Validation-selected stage:",report["selected_stage"],report["validation_losses"])
print("Reload differences:",report["sft_reload_max_abs_difference"],report["dpo_reload_max_abs_difference"])
print("Response truncations:",report["truncated_response_counts"])
for row in report["examples"][:3]:print(json.dumps(row,ensure_ascii=False))


## Interpretation and attribution

Small SFT/preference budgets, one seed, synthetic preference labels and truncated responses. Pairwise log-probability rates and implicit DPO margins are not generated-response win rates, task correctness or human-validated quality. This is a separate alignment experiment on the same base architecture, not reuse of the earlier CodeAlpaca adapter.

[Argilla UltraFeedback Binarized Preferences Cleaned](https://huggingface.co/datasets/argilla/ultrafeedback-binarized-preferences-cleaned), MIT per its dataset card, pinned to revision770076f077c4c5e298498fa32f804857f46d5134. Base: [Meta OPT-350M](https://huggingface.co/facebook/opt-350m), with its research/noncommercial terms. [DPO paper](https://arxiv.org/abs/2305.18290). No supplied course checkpoint, pretrained base weights or dataset archive is bundled.

Concepts studied through the [IBM AI Engineering Professional Certificate](https://www.coursera.org/professional-certificates/ai-engineer). Original provider conditions remain applicable. Mechanical execution and reload checks do not replace task-specific or human quality evaluation.